In [1]:
import requests
from pathlib import Path
import pandas as pd
import pyspark
from pyspark.sql import SparkSession, types
from pyspark.sql import functions as F


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"pyspark version: {pyspark.__version__}")
print(f"spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/06 00:21:54 WARN Utils: Your hostname, mballo-pc, resolves to a loopback address: 127.0.1.1; using 192.168.1.94 instead (on interface wlp1s0)
26/03/06 00:21:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 00:21:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


pyspark version: 4.1.1
spark version: 4.1.1


In [3]:
import requests
from pathlib import Path

URL_PREFIX = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download"

def download_taxi_data(taxi_type: str, year: int) -> None:
    errors = []
    for month in range(1, 13):
        fmonth = f"{month:02d}"
        filename = f"{taxi_type}_tripdata_{year}-{fmonth}.csv.gz"
        url = f"{URL_PREFIX}/{taxi_type}/{filename}"
        local_dir = Path(f"data/raw/{taxi_type}/{year}/{fmonth}")
        local_path = local_dir / filename

        if local_path.exists():
            print(f"  skipping {filename} (already exists)")
            continue

        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()

            local_dir.mkdir(parents=True, exist_ok=True)
            with open(local_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"  ✓ {filename}")

        except requests.exceptions.HTTPError as e:
            print(f"  ✗ {filename} — HTTP error: {e}")
            errors.append((taxi_type, year, month, str(e)))
        except requests.exceptions.ConnectionError:
            print(f"  ✗ {filename} — connection error, skipping")
            errors.append((taxi_type, year, month, "connection error"))
        except Exception as e:
            print(f"  ✗ {filename} — unexpected error: {e}")
            errors.append((taxi_type, year, month, str(e)))

    if errors:
        print(f"\n⚠ {len(errors)} erreur(s) pour {taxi_type}/{year}:")
        for _, _, m, msg in errors:
            print(f"    mois {m:02d}: {msg}")


In [4]:
download_taxi_data("green", 2020)
download_taxi_data("yellow", 2020)

  skipping green_tripdata_2020-01.csv.gz (already exists)
  skipping green_tripdata_2020-02.csv.gz (already exists)
  skipping green_tripdata_2020-03.csv.gz (already exists)
  skipping green_tripdata_2020-04.csv.gz (already exists)
  skipping green_tripdata_2020-05.csv.gz (already exists)
  skipping green_tripdata_2020-06.csv.gz (already exists)
  skipping green_tripdata_2020-07.csv.gz (already exists)
  skipping green_tripdata_2020-08.csv.gz (already exists)
  skipping green_tripdata_2020-09.csv.gz (already exists)
  skipping green_tripdata_2020-10.csv.gz (already exists)
  skipping green_tripdata_2020-11.csv.gz (already exists)
  skipping green_tripdata_2020-12.csv.gz (already exists)
  skipping yellow_tripdata_2020-01.csv.gz (already exists)
  skipping yellow_tripdata_2020-02.csv.gz (already exists)
  skipping yellow_tripdata_2020-03.csv.gz (already exists)
  skipping yellow_tripdata_2020-04.csv.gz (already exists)
  skipping yellow_tripdata_2020-05.csv.gz (already exists)
  skippin

In [5]:
download_taxi_data("green", 2021)
download_taxi_data("yellow", 2021)

  skipping green_tripdata_2021-01.csv.gz (already exists)
  skipping green_tripdata_2021-02.csv.gz (already exists)
  skipping green_tripdata_2021-03.csv.gz (already exists)
  skipping green_tripdata_2021-04.csv.gz (already exists)
  skipping green_tripdata_2021-05.csv.gz (already exists)
  skipping green_tripdata_2021-06.csv.gz (already exists)
  skipping green_tripdata_2021-07.csv.gz (already exists)
  ✗ green_tripdata_2021-08.csv.gz — HTTP error: 404 Client Error: Not Found for url: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-08.csv.gz
  ✗ green_tripdata_2021-09.csv.gz — HTTP error: 404 Client Error: Not Found for url: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-09.csv.gz
  ✗ green_tripdata_2021-10.csv.gz — HTTP error: 404 Client Error: Not Found for url: https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2021-10.csv.gz
  ✗ green_tripdata_2021-11.csv.gz — HTT

In [6]:
# Lire sans schéma pour voir les vraies colonnes
df_check = spark.read \
    .option("header", "true") \
    .csv('./data/raw/yellow/2020/01/yellow_tripdata_2020-01.csv.gz')

print(df_check.columns)
print(len(df_check.columns))

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']
18


In [7]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [8]:
# Lire sans schéma pour voir les vraies colonnes
df_check = spark.read \
    .option("header", "true") \
    .csv('./data/raw/green/2020/01/green_tripdata_2020-01.csv.gz')

print(df_check.columns)
print(len(df_check.columns))

['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge']
20


In [9]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [10]:
schemas = {
    "green": green_schema,
    "yellow": yellow_schema
}
taxi_types=["green", "yellow"]

In [11]:
def csv_to_parquet(spark, taxi_types, year, schemas):
    errors = []
    for taxi in taxi_types:
        for month in range(1, 13):
            print(f"processing data for {taxi}/{year}/{month:02d}")

            input_file = f"data/raw/{taxi}/{year}/{month:02d}/{taxi}_tripdata_{year}-{month:02d}.csv.gz"
            output_path = f"data/pq/{taxi}/{year}/{month:02d}/"

            if not Path(input_file).exists():
                print(f"  ✗ fichier source absent, skipping")
                errors.append((taxi, year, month, "fichier source absent"))
                continue

            if Path(output_path).exists():
                print(f"  skipping, already exists")
                continue

            try:
                df = spark.read \
                    .option("header", "true") \
                    .schema(schemas[taxi]) \
                    .csv(input_file)

                df.repartition(2) \
                  .write.mode("overwrite") \
                  .parquet(output_path)

                print(f"  ✓ saved to {output_path}")

            except Exception as e:
                print(f"  ✗ erreur lors du traitement: {e}")
                errors.append((taxi, year, month, str(e)))

    if errors:
        print(f"\n⚠ {len(errors)} erreur(s):")
        for taxi, year, m, msg in errors:
            print(f"    {taxi}/{year}/{m:02d}: {msg}")

        

In [12]:
csv_to_parquet(
    spark=spark,
    taxi_types=taxi_types,
    year=2020,
    schemas=schemas
)

processing data for green/2020/01
  skipping, already exists
processing data for green/2020/02
  skipping, already exists
processing data for green/2020/03
  skipping, already exists
processing data for green/2020/04
  skipping, already exists
processing data for green/2020/05
  skipping, already exists
processing data for green/2020/06
  skipping, already exists
processing data for green/2020/07
  skipping, already exists
processing data for green/2020/08
  skipping, already exists
processing data for green/2020/09
  skipping, already exists
processing data for green/2020/10
  skipping, already exists
processing data for green/2020/11
  skipping, already exists
processing data for green/2020/12
  skipping, already exists
processing data for yellow/2020/01
  skipping, already exists
processing data for yellow/2020/02
  skipping, already exists
processing data for yellow/2020/03
  skipping, already exists
processing data for yellow/2020/04
  skipping, already exists
processing data for 

In [13]:
csv_to_parquet(
    spark=spark,
    taxi_types=taxi_types,
    year=2021,
    schemas=schemas
)

processing data for green/2021/01
  skipping, already exists
processing data for green/2021/02
  skipping, already exists
processing data for green/2021/03
  skipping, already exists
processing data for green/2021/04
  skipping, already exists
processing data for green/2021/05
  skipping, already exists
processing data for green/2021/06
  skipping, already exists
processing data for green/2021/07
  skipping, already exists
processing data for green/2021/08
  ✗ fichier source absent, skipping
processing data for green/2021/09
  ✗ fichier source absent, skipping
processing data for green/2021/10
  ✗ fichier source absent, skipping
processing data for green/2021/11
  ✗ fichier source absent, skipping
processing data for green/2021/12
  ✗ fichier source absent, skipping
processing data for yellow/2021/01


  ✓ saved to data/pq/yellow/2021/01/
processing data for yellow/2021/02


  ✓ saved to data/pq/yellow/2021/02/
processing data for yellow/2021/03


  ✓ saved to data/pq/yellow/2021/03/
processing data for yellow/2021/04


  ✓ saved to data/pq/yellow/2021/04/
processing data for yellow/2021/05


  ✓ saved to data/pq/yellow/2021/05/
processing data for yellow/2021/06


  ✓ saved to data/pq/yellow/2021/06/
processing data for yellow/2021/07


[Stage 22:>                                                         (0 + 2) / 2]

  ✓ saved to data/pq/yellow/2021/07/
processing data for yellow/2021/08
  ✗ fichier source absent, skipping
processing data for yellow/2021/09
  ✗ fichier source absent, skipping
processing data for yellow/2021/10
  ✗ fichier source absent, skipping
processing data for yellow/2021/11
  ✗ fichier source absent, skipping
processing data for yellow/2021/12
  ✗ fichier source absent, skipping

⚠ 10 erreur(s):
    green/2021/08: fichier source absent
    green/2021/09: fichier source absent
    green/2021/10: fichier source absent
    green/2021/11: fichier source absent
    green/2021/12: fichier source absent
    yellow/2021/08: fichier source absent
    yellow/2021/09: fichier source absent
    yellow/2021/10: fichier source absent
    yellow/2021/11: fichier source absent
    yellow/2021/12: fichier source absent


In [17]:
df_green = spark.read.format('parquet').option('header', 'true').load('./data/pq/green/*/*')

26/03/06 00:34:37 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ./data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

In [19]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

26/03/06 00:36:01 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/yellow/*/*.
java.io.FileNotFoundException: File data/pq/yellow/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

In [18]:
import os
# Vérifier si le dossier existe
print(os.path.exists('data/pq/green/'))

# Lister ce qui est disponible
for root, dirs, files in os.walk('data/pq'):
    print(root)

True
data/pq
data/pq/green
data/pq/green/2021
data/pq/green/2021/06
data/pq/green/2021/02
data/pq/green/2021/07
data/pq/green/2021/04
data/pq/green/2021/03
data/pq/green/2021/01
data/pq/green/2021/05
data/pq/green/2020
data/pq/green/2020/09
data/pq/green/2020/08
data/pq/green/2020/12
data/pq/green/2020/10
data/pq/green/2020/06
data/pq/green/2020/02
data/pq/green/2020/07
data/pq/green/2020/11
data/pq/green/2020/04
data/pq/green/2020/03
data/pq/green/2020/01
data/pq/green/2020/05
data/pq/yellow
data/pq/yellow/2021
data/pq/yellow/2021/06
data/pq/yellow/2021/02
data/pq/yellow/2021/07
data/pq/yellow/2021/04
data/pq/yellow/2021/03
data/pq/yellow/2021/01
data/pq/yellow/2021/05
data/pq/yellow/2020
data/pq/yellow/2020/09
data/pq/yellow/2020/08
data/pq/yellow/2020/12
data/pq/yellow/2020/10
data/pq/yellow/2020/06
data/pq/yellow/2020/02
data/pq/yellow/2020/07
data/pq/yellow/2020/11
data/pq/yellow/2020/04
data/pq/yellow/2020/03
data/pq/yellow/2020/01
data/pq/yellow/2020/05
